# Energy Analyst Programming Exercise — working notebook
Work happens here first; approved code is exported to `main.py`.

## Parameters
System-specific parameters (file paths, packages). Update these to run on another machine.

In [1]:
# Required packages: pandas, numpy, matplotlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()                          # repo root (main.py uses its own folder)
DATA_DIR = BASE_DIR / "historicalPriceData"   # historical power price data
OUTPUT_DIR = BASE_DIR / "output"              # all generated CSVs and images
OUTPUT_DIR.mkdir(exist_ok=True)

## Task 1 — Read and combine all historical data files

In [2]:
# Read every yearly price file and stack them into one dataframe
files = sorted(DATA_DIR.glob("*.csv"))
prices = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# Parse timestamps so we can group by year/month and sort by hour later
prices["Date"] = pd.to_datetime(prices["Date"])

# Quick checks
print("Files read:", [f.name for f in files])
print("Rows:", len(prices))
print("Date range:", prices["Date"].min(), "to", prices["Date"].max())
print("Settlement points:", prices["SettlementPoint"].nunique())
print(sorted(prices["SettlementPoint"].unique()))
print("Missing prices:", prices["Price"].isna().sum())
prices.head()

Files read: ['ERCOT_DA_Prices_2016.csv', 'ERCOT_DA_Prices_2017.csv', 'ERCOT_DA_Prices_2018.csv', 'ERCOT_DA_Prices_2019.csv']
Rows: 497320
Date range: 2016-01-01 00:00:00 to 2019-12-31 23:00:00
Settlement points: 15
['HB_BUSAVG', 'HB_HOUSTON', 'HB_HUBAVG', 'HB_NORTH', 'HB_PAN', 'HB_SOUTH', 'HB_WEST', 'LZ_AEN', 'LZ_CPS', 'LZ_HOUSTON', 'LZ_LCRA', 'LZ_NORTH', 'LZ_RAYBN', 'LZ_SOUTH', 'LZ_WEST']
Missing prices: 0


,Date,SettlementPoint,Price
0,2016-01-01,HB_BUSAVG,18.42
1,2016-01-01,HB_HOUSTON,18.42
2,2016-01-01,HB_HUBAVG,18.42
3,2016-01-01,HB_NORTH,18.43
4,2016-01-01,HB_SOUTH,18.38


## Task 2 — Monthly average price per settlement point (Jan 2016–Dec 2019, keep prices ≤ 0)

In [3]:
# Add Year and Month columns from the hourly timestamp
prices["Year"] = prices["Date"].dt.year
prices["Month"] = prices["Date"].dt.month

# Average price per settlement point per year-month.
# Prices <= 0 are kept on purpose: negative prices are real in deregulated markets.
# Result: 681 rows, not 15 x 48 = 720. HB_PAN only has data from 2019-04-06,
# so it has 9 year-months (Apr-Dec 2019); April 2019 is a partial month (600 hours).
monthly_avg = (
    prices.groupby(["SettlementPoint", "Year", "Month"], as_index=False)["Price"]
    .mean()
    .rename(columns={"Price": "AveragePrice"})
)

# Quick checks
print("Rows:", len(monthly_avg))
print("Year-months per settlement point:")
print(monthly_avg.groupby("SettlementPoint").size().to_string())
print("Lowest monthly average:\n", monthly_avg.nsmallest(1, "AveragePrice").to_string(index=False))
print("Highest monthly average:\n", monthly_avg.nlargest(1, "AveragePrice").to_string(index=False))
monthly_avg.head()

Rows: 681
Year-months per settlement point:
SettlementPoint
HB_BUSAVG     48
HB_HOUSTON    48
HB_HUBAVG     48
HB_NORTH      48
HB_PAN         9
HB_SOUTH      48
HB_WEST       48
LZ_AEN        48
LZ_CPS        48
LZ_HOUSTON    48
LZ_LCRA       48
LZ_NORTH      48
LZ_RAYBN      48
LZ_SOUTH      48
LZ_WEST       48
Lowest monthly average:
 SettlementPoint  Year  Month  AveragePrice
         HB_PAN  2019     12      8.817245
Highest monthly average:
 SettlementPoint  Year  Month  AveragePrice
        LZ_WEST  2019      8    128.876465


,SettlementPoint,Year,Month,AveragePrice
0,HB_BUSAVG,2016,1,19.209301
1,HB_BUSAVG,2016,2,15.288247
2,HB_BUSAVG,2016,3,16.755626
3,HB_BUSAVG,2016,4,18.289347
4,HB_BUSAVG,2016,5,18.159583


In [4]:
# Second average: one price per year-month across ALL settlement points (48 rows).
# Uses every hourly price from every point in that month, so it weights by hours of data.
# Note: HB_PAN joins from April 2019, so from then on 15 points feed the average instead of 14.
monthly_avg_all_points = (
    prices.groupby(["Year", "Month"], as_index=False)["Price"]
    .mean()
    .rename(columns={"Price": "AveragePrice"})
)

# Quick checks
print("Rows:", len(monthly_avg_all_points))
print("Lowest:\n", monthly_avg_all_points.nsmallest(1, "AveragePrice").to_string(index=False))
print("Highest:\n", monthly_avg_all_points.nlargest(1, "AveragePrice").to_string(index=False))
# Compare with the simple mean of the per-point monthly averages
alt = monthly_avg.groupby(["Year", "Month"])["AveragePrice"].mean().values
print("Max difference vs mean of per-point averages:", abs(monthly_avg_all_points["AveragePrice"].values - alt).max())
monthly_avg_all_points.head()

Rows: 48
Lowest:
  Year  Month  AveragePrice
 2016      2     15.387485
Highest:
  Year  Month  AveragePrice
 2019      8     126.67975
Max difference vs mean of per-point averages: 0.07176926966292285


,Year,Month,AveragePrice
0,2016,1,19.335748
1,2016,2,15.387485
2,2016,3,16.991993
3,2016,4,18.557516
4,2016,5,18.501367


## Task 3 — Write AveragePriceByMonth.csv (SettlementPoint, Year, Month, AveragePrice)

In [5]:
# Write the per-settlement-point monthly averages (exactly the 4 required columns, no index)
monthly_avg[["SettlementPoint", "Year", "Month", "AveragePrice"]].to_csv(
    OUTPUT_DIR / "AveragePriceByMonth.csv", index=False
)

# Quick checks
check = pd.read_csv(OUTPUT_DIR / "AveragePriceByMonth.csv")
print("Columns:", list(check.columns))
print("Rows:", len(check))
print(check.head(3).to_string(index=False))

Columns: ['SettlementPoint', 'Year', 'Month', 'AveragePrice']
Rows: 681
SettlementPoint  Year  Month  AveragePrice
      HB_BUSAVG  2016      1     19.209301
      HB_BUSAVG  2016      2     15.288247
      HB_BUSAVG  2016      3     16.755626


## Task 4 — Hourly volatility per year, hubs (HB_) only, set prices ≤ 0 to NaN before log returns

In [6]:
# Hubs only (prefix HB_); load zones are excluded by the task
hubs = prices[prices["SettlementPoint"].str.startswith("HB_")].copy()

# Sort each hub's prices in time order before taking returns
hubs = hubs.sort_values(["SettlementPoint", "Date"])

# Filter out prices <= 0 (log is only defined for positive values).
# Set them to NaN instead of dropping the rows, so no return is computed across
# the gap (a multi-hour change would otherwise be reported as a 1-hour return).
hubs["Price"] = hubs["Price"].where(hubs["Price"] > 0)

# Hourly log return = ln(P_t) - ln(P_t-1), computed within each hub and year
# (groupby keeps returns from crossing hubs or year boundaries).
# Returns into or out of a NaN hour are NaN and are skipped by std().
hubs["LogReturn"] = hubs.groupby(["SettlementPoint", "Year"])["Price"].transform(
    lambda p: np.log(p).diff()
)

# Hourly volatility = standard deviation of log returns (sample std, ddof=1)
hourly_vol = (
    hubs.groupby(["SettlementPoint", "Year"], as_index=False)["LogReturn"]
    .std()
    .rename(columns={"LogReturn": "HourlyVolatility"})
)

# Quick checks
print("Rows:", len(hourly_vol), "(HB_PAN only has 2019)")
print("Prices set to NaN:", hubs["Price"].isna().sum())
print(hourly_vol.pivot(index="SettlementPoint", columns="Year", values="HourlyVolatility").round(6).to_string())

Rows: 25 (HB_PAN only has 2019)
Prices set to NaN: 1024
Year                 2016      2017      2018      2019
SettlementPoint                                        
HB_BUSAVG        0.185773  0.154092  0.200254  0.249378
HB_HOUSTON       0.191099  0.170477  0.200244  0.247946
HB_HUBAVG        0.187957  0.158687  0.202121  0.250648
HB_NORTH         0.184181  0.152575  0.200159  0.249618
HB_PAN                NaN       NaN       NaN  0.547431
HB_SOUTH         0.208533  0.156145  0.198684  0.248980
HB_WEST          0.206858  0.239376  0.287354  0.325545


## Task 5 — Write HourlyVolatilityByYear.csv (SettlementPoint, Year, HourlyVolatility)

In [7]:
# Write hourly volatilities (exact, case-sensitive column names, no index)
hourly_vol[["SettlementPoint", "Year", "HourlyVolatility"]].to_csv(
    OUTPUT_DIR / "HourlyVolatilityByYear.csv", index=False
)

# Quick checks
check = pd.read_csv(OUTPUT_DIR / "HourlyVolatilityByYear.csv")
print("Columns:", list(check.columns))
print("Rows:", len(check), "| Missing:", check.isna().sum().sum())
print(check.head(3).to_string(index=False))

Columns: ['SettlementPoint', 'Year', 'HourlyVolatility']
Rows: 25 | Missing: 0
SettlementPoint  Year  HourlyVolatility
      HB_BUSAVG  2016          0.185773
      HB_BUSAVG  2017          0.154092
      HB_BUSAVG  2018          0.200254


## Task 6 — Highest-volatility hub per year → MaxVolatilityByYear.csv

## Task 7 — Model-ready files: output/formattedSpotHistory/spot_<SettlementPoint>.csv (15 files)

## Bonus — Mean plots: SettlementHubAveragePriceByMonth.png, LoadZoneAveragePriceByMonth.png

## Bonus — Volatility plots

## Bonus — Hourly shape profiles: output/hourlyShapeProfiles/profile_<SettlementPoint>.csv (15 files)

## Bonus — Open-ended analysis